In [ ]:
import os
import json
import math
import matplotlib.pyplot as plt
import matplotlib.image as mpimg


def create_figure_grid(
    base_directory,
    figure_rel_path="figures/discrepancy_diagnostics_realspace.png",
    config_filename="used_config.json",
    text_keys=[
        ("calibration_settings", "kappa_prior_vals"),
        ("calibration_settings", "delta_eta_prior_vals"),
    ],
    ncols=4,
    figsize=(16, 12),
    output_path="grid_summary.png"
):
    """
    Create a grid of figures from config directories with overlaid text.

    Parameters
    ----------
    base_directory : str
        Path containing config_XXXX directories

    figure_rel_path : str
        Relative path inside each config directory to the figure

    config_filename : str
        Name of JSON file containing configuration

    text_keys : list of tuples
        Keys to extract from JSON in hierarchical form
        Example: [("calibration_settings", "kappa_prior_vals")]

    ncols : int
        Number of columns in grid

    figsize : tuple
        Size of the full figure

    output_path : str
        Path to save the resulting grid image
    """

    # --- find config directories ---
    config_dirs = sorted([
        d for d in os.listdir(base_directory)
        if d.startswith("config_") and os.path.isdir(os.path.join(base_directory, d))
    ])

    images = []
    labels = []

    for config_dir in config_dirs:
        full_dir = os.path.join(base_directory, config_dir)

        fig_path = os.path.join(full_dir, figure_rel_path)
        config_path = os.path.join(full_dir, config_filename)

        # --- skip missing files ---
        if not os.path.exists(fig_path) or not os.path.exists(config_path):
            continue

        # --- load image ---
        img = mpimg.imread(fig_path)

        # --- load config ---
        with open(config_path, "r") as f:
            config = json.load(f)

        # --- extract text ---
        text_lines = [config_dir]

        for key_tuple in text_keys:
            val = config
            try:
                for k in key_tuple:
                    val = val[k]
                text_lines.append(f"{key_tuple[-1]}: {val}")
            except KeyError:
                text_lines.append(f"{key_tuple[-1]}: N/A")

        label_text = "\n".join(text_lines)

        images.append(img)
        labels.append(label_text)

    # --- grid layout ---
    n_images = len(images)
    nrows = math.ceil(n_images / ncols)

    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)

    # flatten axes safely
    axes = axes.flatten() if n_images > 1 else [axes]

    for i, ax in enumerate(axes):
        if i < n_images:
            ax.imshow(images[i])
            ax.axis("off")

            # overlay text
            ax.text(
                0.02, 0.98,
                labels[i],
                transform=ax.transAxes,
                fontsize=8,
                verticalalignment='top',
                bbox=dict(facecolor='white', alpha=0.7, edgecolor='none')
            )
        else:
            ax.axis("off")

    plt.tight_layout()
    plt.savefig(output_path, dpi=200)
    plt.close()

    print(f"Saved grid figure to {output_path}")